In [3]:
from pathlib import Path
import re
import pandas as pd
import fitz  # PyMuPDF
from datetime import datetime
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from tqdm import tqdm
import os


# =========================================================
# CONFIG
# =========================================================

PDF_ROOT = Path("/media/hello/Vault/Tribunals/ET_Cases/")
OUTPUT_CSV = Path("/media/hello/Vault/Tribunals/et_sent_to_parties_dates.csv")

# Most ET date headers are near the front
FIRST_N_PAGES = 3

# Concurrency options
USE_PROCESS_POOL = True   # True = faster for large batches on strong CPU
MAX_WORKERS = max(1, (os.cpu_count() or 4) - 1)

# tqdm description
TQDM_DESC = "Scanning PDFs"


# =========================================================
# REGEX
# =========================================================

SENT_TO_PARTIES_REGEX = re.compile(
    r'(?is)'
    r'(sent\s+to\s+(the\s+)?parties\s+on|date\s+sent\s+to\s+(the\s+)?parties)'
    r'\s*[:\-]?\s*'
    r'('
    r'\d{1,2}[\/\-]\d{1,2}[\/\-]\d{2,4}'                # 10/03/2023 or 10-03-2023
    r'|'
    r'\d{1,2}\s+[A-Za-z]+\s+\d{2,4}'                   # 10 March 2023
    r')'
)


# =========================================================
# HELPERS
# =========================================================

def normalize_whitespace(text: str) -> str:
    """
    Collapse repeated whitespace while preserving enough structure
    for regex extraction.
    """
    if not text:
        return ""
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n+", "\n", text)
    return text.strip()


def extract_text_from_pdf(pdf_path: Path, first_n_pages: int = 3) -> str:
    """
    Extract text from the first N pages of a PDF.
    Returns a single text blob. On failure, returns empty string.
    """
    try:
        chunks = []
        with fitz.open(pdf_path) as doc:
            pages_to_read = min(len(doc), first_n_pages)
            for i in range(pages_to_read):
                try:
                    page = doc[i]
                    chunks.append(page.get_text("text"))
                except Exception:
                    continue
        return "\n".join(chunks)
    except Exception:
        return ""


def extract_sent_to_parties_date(text: str) -> str | None:
    """
    Extract the raw 'Sent to the parties on' date string.
    Returns the matched date string or None.
    """
    if not text:
        return None

    text = normalize_whitespace(text)
    match = SENT_TO_PARTIES_REGEX.search(text)

    if match:
        return match.group(4).strip()

    return None


def parse_date_flexibly(date_str: str) -> datetime | None:
    """
    Parse common ET date formats into datetime.
    Supports:
      - 10/03/2023
      - 10-03-2023
      - 10 March 2023
      - 10 Mar 2023
      - 10 MARCH 2023
      - 10/03/23
      - 10 March 23
    """
    if not date_str:
        return None

    cleaned = re.sub(r"\s+", " ", date_str.strip())
    cleaned = cleaned.replace(".", "/")

    formats = [
        "%d/%m/%Y",
        "%d-%m-%Y",
        "%d/%m/%y",
        "%d-%m-%y",
        "%d %B %Y",
        "%d %b %Y",
        "%d %B %y",
        "%d %b %y",
    ]

    for fmt in formats:
        try:
            return datetime.strptime(cleaned, fmt)
        except ValueError:
            continue

    return None


def extract_case_year_from_date(date_obj: datetime | None) -> int | None:
    """
    Returns year from parsed datetime, else None.
    """
    if date_obj is None:
        return None
    return date_obj.year


def process_pdf_for_sent_date(pdf_path_str: str) -> dict:
    """
    Worker-safe function for concurrent execution.
    Accepts a string path so it is easy to pickle for process pools.
    """
    pdf_path = Path(pdf_path_str)

    text = extract_text_from_pdf(pdf_path, first_n_pages=FIRST_N_PAGES)
    raw_date = extract_sent_to_parties_date(text)
    parsed_date = parse_date_flexibly(raw_date) if raw_date else None
    year = extract_case_year_from_date(parsed_date)

    return {
        "file_path": str(pdf_path),
        "file_name": pdf_path.name,
        "sent_to_parties_raw": raw_date,
        "sent_to_parties_iso": parsed_date.strftime("%Y-%m-%d") if parsed_date else None,
        "sent_to_parties_year": year,
        "date_found": raw_date is not None,
    }


def scan_folder_for_sent_dates(pdf_root: Path) -> pd.DataFrame:
    """
    Scan all PDFs under pdf_root recursively and return a DataFrame.
    Uses concurrency + tqdm.
    """
    pdf_files = sorted(pdf_root.rglob("*.pdf"))
    pdf_file_strs = [str(p) for p in pdf_files]

    print(f"Found {len(pdf_files):,} PDF files.")
    print(f"Using {'ProcessPoolExecutor' if USE_PROCESS_POOL else 'ThreadPoolExecutor'} "
          f"with {MAX_WORKERS} workers.")

    if not pdf_files:
        return pd.DataFrame(
            columns=[
                "file_path",
                "file_name",
                "sent_to_parties_raw",
                "sent_to_parties_iso",
                "sent_to_parties_year",
                "date_found",
            ]
        )

    executor_cls = ProcessPoolExecutor if USE_PROCESS_POOL else ThreadPoolExecutor
    rows = []

    with executor_cls(max_workers=MAX_WORKERS) as executor:
        results = executor.map(process_pdf_for_sent_date, pdf_file_strs, chunksize=50 if USE_PROCESS_POOL else 1)
        for row in tqdm(results, total=len(pdf_file_strs), desc=TQDM_DESC):
            rows.append(row)

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df.sort_values(
            by=["sent_to_parties_iso", "file_name"],
            ascending=[True, True],
            na_position="last"
        ).reset_index(drop=True)

    return df


def main():
    if not PDF_ROOT.exists():
        raise FileNotFoundError(f"PDF root not found: {PDF_ROOT}")

    df = scan_folder_for_sent_dates(PDF_ROOT)

    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUTPUT_CSV, index=False)

    print("\nDone.")
    print(f"Output saved to: {OUTPUT_CSV}")
    print(f"Rows: {len(df):,}")

    if "date_found" in df.columns:
        found = int(df["date_found"].sum())
        print(f"Dates found: {found:,}")
        print(f"Dates missing: {len(df) - found:,}")

    if "sent_to_parties_year" in df.columns and not df.empty:
        print("\nYear distribution:")
        print(df["sent_to_parties_year"].value_counts(dropna=False).sort_index())


if __name__ == "__main__":
    main()

Found 127,755 PDF files.
Using ProcessPoolExecutor with 47 workers.


Scanning PDFs: 100%|██████████| 127755/127755 [00:14<00:00, 8615.01it/s]



Done.
Output saved to: /media/hello/Vault/Tribunals/et_sent_to_parties_dates.csv
Rows: 127,755
Dates found: 39,952
Dates missing: 87,803

Year distribution:
sent_to_parties_year
222.0         1
1201.0        1
2002.0        4
2010.0        2
2011.0        3
2012.0        1
2013.0        5
2014.0        2
2015.0       23
2016.0      107
2017.0     3131
2018.0     3970
2019.0     7107
2020.0     5109
2021.0     3751
2022.0     4241
2023.0     4272
2024.0     3980
2025.0     4202
2032.0        2
2034.0        1
2101.0        1
2109.0        1
2201.0        1
2202.0        1
2203.0        1
2924.0        1
3017.0        1
3019.0        1
NaN       87832
Name: count, dtype: int64


In [4]:

from pathlib import Path
import os
import re
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

import fitz  # PyMuPDF
import pandas as pd
from tqdm import tqdm


# =========================================================
# CONFIG
# =========================================================

PDF_ROOT = Path("/media/hello/Vault/Tribunals/ET_Cases/")
OUTPUT_CSV = Path("/media/hello/Vault/Tribunals/et_laddie_hits.csv")

SEARCH_TERM = "Laddie"
SNIPPET_CHARS = 120

# Set to None to scan all pages, or e.g. 5 to scan only first 5 pages.
MAX_PAGES_PER_PDF = None

# Concurrency
USE_PROCESS_POOL = True
MAX_WORKERS = max(1, (os.cpu_count() or 4) - 1)
CHUNKSIZE = 50 if USE_PROCESS_POOL else 1


# =========================================================
# REGEX
# =========================================================

SEARCH_REGEX = re.compile(rf"\b{re.escape(SEARCH_TERM)}\b", re.IGNORECASE)


# =========================================================
# HELPERS
# =========================================================

def normalize_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def extract_snippet(text: str, match_start: int, match_end: int, snippet_chars: int = 120) -> str:
    """
    Return a snippet around the match.
    """
    left = max(0, match_start - snippet_chars)
    right = min(len(text), match_end + snippet_chars)
    return text[left:right].strip()


def search_pdf_for_term(
    pdf_path: Path,
    search_regex: re.Pattern,
    max_pages: int | None = None,
    search_term: str = SEARCH_TERM,
    snippet_chars: int = SNIPPET_CHARS,
) -> list[dict]:
    """
    Search one PDF for a regex term. Returns a list of hit dicts.
    """
    hits: list[dict] = []

    try:
        with fitz.open(pdf_path) as doc:
            total_pages = len(doc)
            pages_to_scan = total_pages if max_pages is None else min(total_pages, max_pages)

            for page_index in range(pages_to_scan):
                try:
                    page = doc[page_index]
                    text = normalize_text(page.get_text("text"))

                    if not text:
                        continue

                    for match in search_regex.finditer(text):
                        snippet = extract_snippet(
                            text=text,
                            match_start=match.start(),
                            match_end=match.end(),
                            snippet_chars=snippet_chars,
                        )

                        hits.append(
                            {
                                "file_path": str(pdf_path),
                                "file_name": pdf_path.name,
                                "page_number": page_index + 1,
                                "search_term": search_term,
                                "matched_text": match.group(0),
                                "snippet": snippet,
                            }
                        )

                except Exception:
                    continue

    except Exception:
        return hits

    return hits


def process_pdf_path(pdf_path_str: str) -> list[dict]:
    """
    Worker-safe wrapper for concurrent execution.
    """
    pdf_path = Path(pdf_path_str)
    return search_pdf_for_term(
        pdf_path=pdf_path,
        search_regex=SEARCH_REGEX,
        max_pages=MAX_PAGES_PER_PDF,
        search_term=SEARCH_TERM,
        snippet_chars=SNIPPET_CHARS,
    )


def scan_folder_for_term(pdf_root: Path) -> pd.DataFrame:
    """
    Scan all PDFs recursively for SEARCH_TERM and return a DataFrame of hits.
    """
    pdf_files = sorted(pdf_root.rglob("*.pdf"))
    pdf_file_strs = [str(p) for p in pdf_files]

    print(f"Found {len(pdf_files):,} PDF files.")
    print(
        f"Using {'ProcessPoolExecutor' if USE_PROCESS_POOL else 'ThreadPoolExecutor'} "
        f"with {MAX_WORKERS} workers."
    )

    if not pdf_files:
        return pd.DataFrame(
            columns=[
                "file_path",
                "file_name",
                "page_number",
                "search_term",
                "matched_text",
                "snippet",
            ]
        )

    executor_cls = ProcessPoolExecutor if USE_PROCESS_POOL else ThreadPoolExecutor
    rows: list[dict] = []

    with executor_cls(max_workers=MAX_WORKERS) as executor:
        results = executor.map(process_pdf_path, pdf_file_strs, chunksize=CHUNKSIZE)
        for hit_list in tqdm(results, total=len(pdf_file_strs), desc=f"Scanning '{SEARCH_TERM}'"):
            if hit_list:
                rows.extend(hit_list)

    df = pd.DataFrame(rows)

    if not df.empty:
        df = df.sort_values(
            by=["file_name", "page_number"],
            ascending=[True, True]
        ).reset_index(drop=True)

    return df


def main():
    if not PDF_ROOT.exists():
        raise FileNotFoundError(f"PDF root not found: {PDF_ROOT}")

    df = scan_folder_for_term(PDF_ROOT)

    OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUTPUT_CSV, index=False)

    print("\nDone.")
    print(f"Output saved to: {OUTPUT_CSV}")
    print(f"Hits: {len(df):,}")


if __name__ == "__main__":
    main()

Found 127,755 PDF files.
Using ProcessPoolExecutor with 47 workers.


Scanning 'Laddie': 100%|██████████| 127755/127755 [00:29<00:00, 4298.03it/s]



Done.
Output saved to: /media/hello/Vault/Tribunals/et_laddie_hits.csv
Hits: 178
